In [2]:
from spark_session import get_spark

In [1]:
import sources

In [3]:
spark = get_spark("spark_sql")
sources.orders(spark).createOrReplaceTempView("orders")
sources.products(spark).createOrReplaceTempView("products")
sources.customers(spark).createOrReplaceTempView("customers")

In [4]:
city_revenue = spark.sql("""
                         WITH clean_customers AS (
                             SELECT DISTINCT customer_id, city FROM customers
                         ), sales AS (
                             SELECT O.customer_id, o.quantity * p.price AS revenue
                             FROM orders o JOIN products p ON o.product_id = o.product_id
                             WHERE o.quantity > 0 AND o.status <> 'refunded'
                         )
                         SELECT c.city, COUNT(*) AS orders, ROUND(SUM(s.revenue), 0) AS revenue
                         FROM sales s JOIN clean_customers c ON s.customer_id = c.customer_id
                         GROUP BY c.city
                         ORDER BY revenue DESC
                         """)

city_revenue.show()

+-------+-------+------------+
|   city| orders|     revenue|
+-------+-------+------------+
|  Osaka|1956040|8.75465502E8|
| Berlin|1922360|8.62924428E8|
|Nairobi|1788920|8.02324167E8|
| Lahore|1762960| 7.8889626E8|
| Austin|1734720|7.78478211E8|
|   Lima|1727600|7.71392504E8|
+-------+-------+------------+



In [5]:
from pyspark.sql import functions as F

In [6]:
total = city_revenue.agg(F.sum("revenue")).first()[0]
city_revenue.withColumn("share_pct", F.round(F.col("revenue") / total * 100, 1)).show()
spark.sql("SELECT status, COUNT(*) AS n FROM orders GROUP BY status").where("n > 5000").show()

+-------+-------+------------+---------+
|   city| orders|     revenue|share_pct|
+-------+-------+------------+---------+
|  Osaka|1956040|8.75465502E8|     17.9|
| Berlin|1922360|8.62924428E8|     17.7|
|Nairobi|1788920|8.02324167E8|     16.4|
| Lahore|1762960| 7.8889626E8|     16.2|
| Austin|1734720|7.78478211E8|     16.0|
|   Lima|1727600|7.71392504E8|     15.8|
+-------+-------+------------+---------+

+-------+------+
| status|     n|
+-------+------+
|   paid|210887|
|shipped| 69812|
+-------+------+



In [7]:
city_revenue.createOrReplaceTempView("city_revenue")
spark.sql("SELECT city FROM city_revenue WHERE revenue > (SELECT AVG(revenue) FROM city_revenue)").show()
spark.sql("SELECT * FROM city_revenue").explain()

+------+
|  city|
+------+
| Osaka|
|Berlin|
+------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [revenue#16 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(revenue#16 DESC NULLS LAST, 8), ENSURE_REQUIREMENTS, [plan_id=1958]
      +- HashAggregate(keys=[city#13], functions=[count(1), sum(revenue#17)])
         +- Exchange hashpartitioning(city#13, 8), ENSURE_REQUIREMENTS, [plan_id=1955]
            +- HashAggregate(keys=[city#13], functions=[partial_count(1), partial_sum(revenue#17)])
               +- Project [revenue#17, city#13]
                  +- BroadcastHashJoin [customer_id#2], [customer_id#11], Inner, BuildRight, false, false
                     :- Project [customer_id#2, (cast(quantity#4 as double) * price#10) AS revenue#17]
                     :  +- BroadcastNestedLoopJoin BuildRight, Inner
                     :     :- Project [customer_id#2, quantity#4]
                     :     :  +- Filter (((((isnotnull(quantity#4) AND isnotnull(stat